# Dag 6+7: Online Endpoints, Batch Endpoints + Model Registry

**Eksamensrelevans:** Train and Deploy Models — 25-30% af DP-100 eksamen

**Nøglebegreber:**
- `ManagedOnlineEndpoint`, `ManagedOnlineDeployment` — real-time inferens
- Scoring script: `init()` og `run()` funktioner
- Blue/green deployment og traffic splitting
- `BatchEndpoint`, `BatchDeployment` — asynkron batch-inferens
- Model Registry: registrering, versionering og tagging

**Læringsmål:**
- Oprette og konfigurere et managed online endpoint
- Skrive et scoring script med `init()` og `run()`
- Deploye en model og teste den med `invoke()`
- Forstå blue/green deployment og traffic-fordeling
- Registrere en model i Model Registry med version og tags
- Oprette et batch endpoint og deploye en model til batch-inferens

**Forudsætninger:** Dag 1-5 gennemført. Du har en trænet model fra Dag 5 (Sweep/Pipeline job).

## 1. MLClient
Opret forbindelse til workspace

In [47]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## 2. Model Registry

Før vi kan deploye noget, skal modellen registreres i Azure ML's **Model Registry**. Model Registry giver dig:
- Versionering (v1, v2, ...)
- Tags til at markere status (f.eks. `stage: production`)
- Mulighed for at referere til modellen fra deployments

En model kan registreres fra:
- En lokal sti
- En job-output (via `azureml://jobs/<job_name>/outputs/<output_name>/paths/`)
- Et MLflow run (via `runs:/<run_id>/model`)

> **Eksamenstip:** `Model` entity har typerne `custom_model`, `mlflow_model`, og `triton_model`. MLflow-modeller (logget med `mlflow.sklearn.log_model`) bruger typen `mlflow_model` og understøtter automatisk inferens uden custom scoring script i visse tilfælde.

**Opgave:** Registrer din model fra et tidligere job i Model Registry.
- Find jobnavnet på dit Sweep-job (eller et andet job fra Dag 3/5) i Azure ML Studio
- Brug `Model()` med `type=AssetTypes.CUSTOM_MODEL` eller `MLFLOW_MODEL`
- Tilføj tags: `{"stage": "staging", "dataset": "ibm-churn"}`
- Registrer med `ml_client.models.create_or_update()`

*Hint:* Sti til job-output: `"azureml://jobs/<job_name>/outputs/default/paths/model/"`

*Hint:* `from azure.ai.ml.entities import Model` og `from azure.ai.ml.constants import AssetTypes`

In [48]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# TODO: Erstat <job_name> med dit faktiske jobnavn (find det i Azure ML Studio under Jobs)
job_name = "attrition-mlflow-final"

model = Model(
    name="attrition-mlflow-model",
    path=f"azureml://jobs/{job_name}/outputs/artifacts/paths/model/",
    type = AssetTypes.MLFLOW_MODEL,
    description = "Test model for DP-100 prep",
    tags = {"stage": "staging", "dataset": "ibm-churn"}
)

# TODO: Registrer modellen
registered_model = ml_client.models.create_or_update(model)

print(f"Registreret: {registered_model.name}, version: {registered_model.version}")

Registreret: attrition-mlflow-model, version: 2


## 3. Model versionering og tagging

> **Eksamenstip:** Du kan opdatere tags og description på en registreret model uden at skabe en ny version — brug `ml_client.models.create_or_update()` med samme navn og version, men ændrede tags.

**Opgave A:** List alle versioner af din `churn-model`.

*Hint:* `ml_client.models.list(name="churn-model")`

**Opgave B:** Hent den seneste version og opdater dens tag til `"stage": "production"`.

*Hint:* `ml_client.models.get(name="churn-model", version="1")` — derefter modificer `.tags` og kald `create_or_update()` igen.

In [51]:
# TODO: List alle versioner af churn-model
for m in ml_client.models.list(name="attrition-mlflow-model"):
    print(f"Version: {m.version}, Tags: {m.tags}")

Version: 2, Tags: {'stage': 'production', 'dataset': 'ibm-churn'}
Version: 1, Tags: {'stage': 'production', 'dataset': 'ibm-churn'}


In [50]:
# TODO: Hent version 1 af churn-model
model_v1 = ml_client.models.get(name="attrition-mlflow-model", version="2")

# TODO: Opdater stage-tagget til "production"
model_v1.tags["stage"] = "production"

# TODO: Gem opdateringen
updated = ml_client.models.create_or_update(model_v1)
print(f"Opdaterede tags: {updated.tags}")

Opdaterede tags: {'stage': 'production', 'dataset': 'ibm-churn'}


## 4. Scoring script: `score.py`

Et online endpoint kræver et **scoring script** med to funktioner:
- `init()` — køres ved container-opstart; bruges til at indlæse modellen
- `run(raw_data)` — køres per request; modtager JSON, returnerer JSON

Argumentet til `run()` er en JSON-streng (ikke et dict). Du skal selv parse den med `json.loads()`.

> **Eksamenstip:** `init()` bruger typisk `AZUREML_MODEL_DIR` environment-variablen til at finde model-stien. For MLflow-modeller kan du bruge `mlflow.pyfunc.load_model()`. For custom modeller kan du bruge `joblib.load()` eller `pickle.load()`.

**Opgave:** Udfyld de to funktioner i scoring-scriptet herunder.
- `init()`: indlæs MLflow-modellen fra `AZUREML_MODEL_DIR`
- `run(raw_data)`: parse inputtet, kald `model.predict()`, returner resultatet som JSON

*Hint:* `os.environ["AZUREML_MODEL_DIR"]` giver dig stien til model-mappen.

*Hint:* For en MLflow pyfunc-model: `mlflow.pyfunc.load_model(model_path)` — kald derefter `.predict(df)` med en pandas DataFrame.

In [59]:
%%writefile ../src/score.py
import os
import json
import pandas as pd
import mlflow.pyfunc

model = None


def init():
    global model
    model_dir = os.environ["AZUREML_MODEL_DIR"]
    # Find den faktiske MLmodel-mappe
    for root, dirs, files in os.walk(model_dir):
        if "MLmodel" in files:
            model_dir = root
            break
        
    model = mlflow.pyfunc.load_model(model_dir)

def run(raw_data: str) -> str:
    """Køres per inference-request.

    Args:
        raw_data: JSON-streng med input-data.
            Eksempel: '{"data": [[35, 3, 2, 3, 5, 5000, 1, 0, 0]]}'

    Returns:
        JSON-streng med predictions.
    """
    # TODO: Parse raw_data med json.loads()
    data = json.loads(raw_data)
    
    import numpy as np
    df = pd.DataFrame(data['data'], columns=['Age', 'WorkLifeBalance', 'YearsSinceLastPromotion', 'JobInvolvement', 'YearsAtCompany', 'MonthlyIncome', 'Gender_Male', 'Department_Research & Development', 'Department_Sales'])
    df = df.astype(np.int32)

    # TODO: Kald model.predict() og konverter resultatet til en Python-liste
    predictions = list(model.predict(df))

    # TODO: Returner resultatet som JSON-streng
    return json.dumps(predictions)

Overwriting ../src/score.py


## 5. Online Endpoint

Et **Managed Online Endpoint** er en fuldt administreret, real-time inferens-tjeneste.

Centrale parametre for `ManagedOnlineEndpoint`:
- `name` — globalt unikt navn (bruges i URL-adressen)
- `auth_mode` — `"key"` (API-nøgle) eller `"aml_token"` (AAD-token)

> **Eksamenstip:** Endpoint-navne skal være globalt unikke og må kun indeholde bogstaver, tal og bindestreger. `auth_mode="key"` er simplest og mest brugt i eksamen-scenarier.

**Opgave:** Opret et managed online endpoint.
- Brug et unikt navn, f.eks. `"churn-endpoint-<dine initialer>"`
- Sæt `auth_mode="key"`
- Opret det med `ml_client.online_endpoints.begin_create_or_update().result()`

*Hint:* `from azure.ai.ml.entities import ManagedOnlineEndpoint`

In [60]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

# TODO: Vælg et globalt unikt endpointnavn
endpoint_name = "churn-endpoint-mrg"

# TODO: Opret et ManagedOnlineEndpoint med name og auth_mode="key"
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key"
)

# TODO: Opret endpointet (begin_create_or_update returnerer en poller — kald .result() for at vente)
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Endpoint oprettet: {endpoint.name}, status: {endpoint.provisioning_state}")

Endpoint oprettet: churn-endpoint-mrg, status: Succeeded


## 6. Online Deployment (blue)

Et **deployment** er den konkrete model-instans bag et endpoint. Et endpoint kan have *flere* deployments, hvor traffic fordeles imellem dem.

Centrale parametre for `ManagedOnlineDeployment`:
- `name` — navn på deployment (f.eks. `"blue"`)
- `endpoint_name` — endpoint det tilhører
- `model` — reference til registreret model (f.eks. `"churn-model:1"`)
- `environment` — environment med de nødvendige dependencies
- `code_configuration` — scoring script sti og script-filnavn
- `instance_type` — VM-størrelse (f.eks. `"Standard_DS2_v2"`)
- `instance_count` — antal replikaer

> **Eksamenstip:** `CodeConfiguration` kræver `code` (mappe) og `scoring_script` (filnavn). For MLflow-modeller kan du udelade `code_configuration` — Azure ML genererer automatisk en scorer. Men du skal vide, hvornår og hvorfor du vil skrive din egen.

**Opgave:** Opret et deployment ved navn `"blue"` til dit endpoint.
- Brug din registrerede `churn-model:1`
- Brug `custom-environment@latest`
- Brug `CodeConfiguration(code="../src", scoring_script="score.py")`
- Sæt `instance_type="Standard_DS2_v2"` og `instance_count=1`

*Hint:* `from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration`

In [61]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

# TODO: Opret et ManagedOnlineDeployment
blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model="attrition-mlflow-model:2",
    environment="custom-environment@latest",
    code_configuration = CodeConfiguration(code="../src", scoring_script="score.py"),
    instance_type="STANDARD_D2AS_V4",
    instance_count=1
)

# TODO: Deploy (begin_create_or_update().result())
blue_deployment = ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

print(f"Deployment '{blue_deployment.name}' klar.")

Check: endpoint churn-endpoint-mrg exists
Uploading src (0.02 MBs): 100%|██████████| 15450/15450 [00:00<00:00, 87626.75it/s]




........................................Deployment 'blue' klar.


In [24]:
logs = ml_client.online_deployments.get_logs(                                                                                                                                                         
    name="blue", endpoint_name=endpoint_name, lines=100                                                                                                                                               
)                                                                                                                                                                                                     
print(logs)  

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-03-01T12:32:07.144189Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-03-01T12:32:07.153619Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-03-01T12:32:13.141911Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-03-01T12:32:13.141911Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-03-01T12:32:13.378827Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-03-01T12:32:13.592674Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-03-01T12:32:32.82404174Z, Message: Container is ready

Container logs:
2026-03-01 12:

## 7. Traffic og blue/green deployment

Når du har to deployments (`blue` og `green`), kan du fordele trafik imellem dem. Dette bruges til:
- **Canary releases:** Send 10% trafik til ny model, 90% til gammel
- **A/B testing:** Sammenlign to modellers performance i produktion
- **Zero-downtime updates:** Flyt gradvist trafik til nyt deployment

Traffic-fordelingen sættes direkte på **endpointet** (ikke deployment'et) som en dict:
```python
endpoint.traffic = {"blue": 90, "green": 10}
```
Summen skal altid give 100.

> **Eksamenstip:** Et nyt deployment starter med 0% traffic. Du skal eksplicit opdatere endpoint.traffic for at sende requests til det. Husk at et endpoint altid har et `default` traffic-flow.

**Opgave A:** Sæt 100% trafik til `blue` deployment.

**Opgave B (konceptuelt):** Beskriv i en kommentar, hvad du ville gøre for at lave en blue/green-switch til et hypotetisk `green` deployment.

*Hint:* Opdater `endpoint.traffic` og kald `ml_client.online_endpoints.begin_create_or_update(endpoint).result()`

In [27]:
endpoint = ml_client.online_endpoints.get(name=endpoint_name)

In [28]:
endpoint.traffic

{'blue': 100}

In [63]:
# TODO: Hent det eksisterende endpoint-objekt
endpoint = ml_client.online_endpoints.get(name=endpoint_name)

# TODO: Sæt traffic til 100% blue
endpoint.traffic = {"blue": 100}
print(f"Traffic-fordeling: {endpoint.traffic}")

# TODO: Opdater endpointet
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint)

# Konceptuelt: Hvad ville du gøre for at skifte til green?
# Jeg ville lave et get() kald for at få fat i objektet, derefter bruger object.traffic = {"blue": 90, "green": 10} til at splitte traffik

Traffic-fordeling: {'blue': 100}


Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


## 8. Test endpointet med `invoke()`

Efter deployment kan du teste dit endpoint direkte fra SDK'et.

> **Eksamenstip:** `ml_client.online_endpoints.invoke()` sender et HTTP POST-request til endpointet. Du kan angive et specifikt `deployment_name` for at ramme et bestemt deployment (bypass traffic-fordeling). Uden `deployment_name` følger det traffic-reglerne.

Input-formatet afhænger af dit scoring script. Vores `run()` forventer:
```json
{"data": [[35, 3, 2, 3, 5, 5000, 1, 0, 0]]}
```
Kolonnerne svarer til: ['Age', 'WorkLifeBalance', 'YearsSinceLastPromotion', 'JobInvolvement', 'YearsAtCompany', 'MonthlyIncome', 'Gender_Male', 'Department_Research & Development', 'Department_Sales']

**Opgave:** Test dit endpoint med et JSON-input.
- Lav en test-request med en eller flere rækker
- Brug `ml_client.online_endpoints.invoke()` med `endpoint_name` og `request_file`
- Alternativt: send `input_data` direkte som string (afhænger af SDK-version)

*Hint:* Skriv en JSON-fil til disk med `Path("test_input.json").write_text(...)` og brug `request_file="test_input.json"`

In [65]:
import json
from pathlib import Path

# TODO: Lav et test-input (en eller flere rækker)
test_data = {
    "data": [
        [35, 3, 2, 3, 5, 5000, 1, 0, 0]
    ]
}

# TODO: Skriv til en JSON-fil
request_file = Path("../data/test_input.json")
request_file.write_text(json.dumps(test_data))

# TODO: Kald invoke() og print resultatet
result = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file=request_file
)

print(f"Svar fra endpoint: {result}")

Svar fra endpoint: "[\"No\"]"


## 9. Batch Endpoint

Et **Batch Endpoint** bruges til asynkron scoring af store datamængder. I modsætning til online endpoints starter du et **batch inference job** ved at invoke'e endpointet — resultatet skrives til en output-mappe.

Typiske use cases:
- Scorer en hel måned-database om natten
- Behandler filer fra blob storage i batches

Centrale entiteter:
- `BatchEndpoint` — selve endpointet (navn + auth)
- `BatchDeployment` — modellen + compute + mini-batch settings
- `BatchRetrySettings` — retry ved fejl

Vigtige `BatchDeployment`-parametre:
- `compute` — compute cluster (f.eks. `"my-cluster"`)
- `mini_batch_size` — antal filer per mini-batch
- `output_action` — `"append_row"` (append til CSV) eller `"summary_only"`
- `max_concurrency_per_instance` og `instance_count`

> **Eksamenstip:** Batch deployments kræver compute cluster (AmlCompute), *ikke* managed compute som online endpoints. `mini_batch_size` styrer hvor mange filer der sendes til `run()` ad gangen. `output_action="append_row"` er standard for scoring-output.

**Opgave:** Opret et batch endpoint og et tilhørende deployment.
- Endpoint navn: `"churn-batch-<initialer>"`
- Deployment: brug din registrerede `churn-model:1`, `my-cluster`, og `score.py`
- Sæt `mini_batch_size=10`, `instance_count=1`

*Hint:* `from azure.ai.ml.entities import BatchEndpoint, BatchDeployment, BatchRetrySettings`

*Hint:* `from azure.ai.ml.constants import BatchDeploymentOutputAction`

In [ ]:
from azure.ai.ml.entities import BatchEndpoint, BatchDeployment, BatchRetrySettings
from azure.ai.ml.constants import BatchDeploymentOutputAction

# TODO: Vælg et unikt batch endpoint-navn
batch_endpoint_name = "churn-batch-<initialer>"

# TODO: Opret BatchEndpoint
batch_endpoint = ...

# TODO: Opret endpointet (begin_create_or_update().result())
batch_endpoint = ...

print(f"Batch endpoint: {batch_endpoint.name}")

In [ ]:
# TODO: Opret BatchDeployment med:
#   - name = "batch-blue"
#   - endpoint_name = batch_endpoint_name
#   - model = "churn-model:1"
#   - environment = "custom-environment@latest"
#   - code_configuration = CodeConfiguration(code="../src", scoring_script="score.py")
#   - compute = "my-cluster"
#   - instance_count = 1
#   - mini_batch_size = 10
#   - output_action = BatchDeploymentOutputAction.APPEND_ROW
#   - retry_settings = BatchRetrySettings(max_retries=3, timeout=300)

batch_deployment = ...

# TODO: Deploy
batch_deployment = ...

print(f"Batch deployment '{batch_deployment.name}' klar.")

## 10. Batch Endpoint Invokering

Et batch endpoint invoke'es ved at starte et **batch inference job**. Inputtet peger på en data asset eller en URI.

> **Eksamenstip:** Når du invoke'er et batch endpoint returneres et `BatchJob`-objekt. Du kan monitorere det med `ml_client.jobs.get(job.name)`. Output skrives til den konfigurerede output-sti (default: workspaceblobstore).

**Opgave:** Start et batch inference job mod dit batch endpoint.
- Brug dit `ibm-churn-file:1` data asset som input
- Invoke med `ml_client.batch_endpoints.invoke()`
- Hent job-status med `ml_client.jobs.get(job.name)`

*Hint:* `from azure.ai.ml import Input` — brug `AssetTypes.URI_FILE` for din data asset.

In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# TODO: Definer input til batch job (brug ibm-churn-file:1)
batch_input = Input(
    type=...,
    path=...
)

# TODO: Invoke batch endpointet
#   - endpoint_name = batch_endpoint_name
#   - input = batch_input
batch_job = ml_client.batch_endpoints.invoke(
    ...
)

# TODO: Print job-navn og status
print(f"Batch job startet: {batch_job.name}")

# TODO: Hent den aktuelle status
job_status = ...
print(f"Job status: {job_status.status}")

## 11. Oprydning (vigtigt — koster penge!)

Online endpoints faktureres per time, selv når de ikke modtager trafik. Husk at slette dem efter brug.

> **Eksamenstip:** Du skal slette **deployments** inden du kan slette et **endpoint** med visse SDK-versioner — eller du kan bruge `begin_delete()` direkte på endpointet, som cascade-sletter deployments.

**Opgave:** Slet dit online endpoint og batch endpoint.

*Hint:* `ml_client.online_endpoints.begin_delete(name=endpoint_name).result()`

*Hint:* `ml_client.batch_endpoints.begin_delete(name=batch_endpoint_name).result()`

In [46]:
# TODO: Slet online endpoint (cascade-sletter deployments)
ml_client.online_deployments.begin_delete(name="blue", endpoint_name="churn-endpoint-mrg").result()

# TODO: Slet batch endpoint

print("Endpoints slettet.")

Endpoints slettet.


## 12. Bonus: MLflow autolog-model til endpoint (uden scoring script)

MLflow-modeller logget med `mlflow.sklearn.log_model()` understøtter **no-code deployment** — Azure ML kan generere scoring-scriptet automatisk.

**Opgave:** Opret et nyt deployment til dit (nu slettede) endpoint, men denne gang uden `code_configuration`. Brug blot modellen og et curated MLflow environment.

*Hint:* Curated environment: `"AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest"` — eller find et aktuelt MLflow environment med `ml_client.environments.list()`.

*Hint:* Uden `code_configuration` skal modellen være type `MLFLOW_MODEL`. Azure ML genererer automatisk en scorer baseret på model-signaturen.

In [ ]:
# BONUS: Opret et endpoint og et no-code MLflow deployment
# TODO: Genskab endpoint

# TODO: Opret ManagedOnlineDeployment uden code_configuration
#        - model = "churn-model:1" (skal være MLFLOW_MODEL type)
#        - environment = et curated MLflow/sklearn environment
#        - instance_type, instance_count som før

# TODO: Deploy og sæt 100% traffic

# TODO: Test med invoke() — bemærk at input-format nu følger MLflow-modellens signatur
#        Format: {"input_data": {"columns": [...], "data": [[...]]}} 

## 13. Eksamensquiz

Svar kort på hvert spørgsmål nedenfor. Skriv svarene som kommentarer eller i en markdown-celle.

**Spørgsmål 1:** Hvad er forskellen på `auth_mode="key"` og `auth_mode="aml_token"` for et online endpoint?

Ved `aml_token` skal man nok supplere en API token per request, hvorimod `key` authentikerer via min bruger. 

Ved faktisk ikke hvad forskellen er, giv mig gerne en forklaring.

Men det har self noget at gøre med authentication.

**Spørgsmål 2:** Hvad sker der, hvis din `init()` funktion kaster en exception under opstart?

Så fejler min deployment.

**Spørgsmål 3:** Du har deployments `blue` (80%) og `green` (20%). Hvordan tester du *udelukkende* `green` deployment uden at ændre traffic-fordelingen?

Jeg kan override traffic parametret når jeg laver et invoke()-kald. Der kan jeg specificere deployment.

**Spørgsmål 4:** Hvad er `mini_batch_size` i en BatchDeployment, og hvad sendes til `run()` i et batch scoring script?

Det er størrelsen på de subsets af hele batchet, der scores af gangen.

Batch size på 100 og mini batch size på 20 medfører 5 iterationer af predictions.

**Spørgsmål 5:** Hvornår bruger du `MLFLOW_MODEL` type vs. `CUSTOM_MODEL` type i Model Registry?

Custom models er til atypiske modeller, der gemmes via joblib. 

MLFLOW modeller er modeller fra en kendt flavor, e.g. scikit-learn.

**Spørgsmål 6:** Du registrerer en model med samme navn men udelader version. Hvad sker der?

Der oprettes en ny version.

**Spørgsmål 7:** Online endpoint faktureres pr. ______ mens batch endpoint kun faktureres pr. ______.

Online endpoints faktureres per time, mens batch endpoints faktureres per request.

## Nøglepunkter til eksamen

**Model Registry:**
- `Model(name, path, type, tags)` — registrer med `ml_client.models.create_or_update()`
- Typer: `MLFLOW_MODEL`, `CUSTOM_MODEL`, `TRITON_MODEL`
- Tags og description kan opdateres uden at skabe ny version
- Hent specifik version: `ml_client.models.get(name, version)`
- Hent seneste: `ml_client.models.get(name, label="latest")`

**Online Endpoints:**
- `ManagedOnlineEndpoint(name, auth_mode)` — `auth_mode` er `"key"` eller `"aml_token"`
- `ManagedOnlineDeployment` kræver: `model`, `environment`, `code_configuration`, `instance_type`, `instance_count`
- MLflow-modeller: `code_configuration` er valgfri (no-code deployment)
- Traffic sættes på **endpointet**, ikke deployment: `endpoint.traffic = {"blue": 100}`
- Test: `ml_client.online_endpoints.invoke(endpoint_name, request_file)`
- Angiv `deployment_name` i invoke for at bypass traffic-fordeling

**Scoring script:**
- `init()` køres ved containerstart — indlæs model her
- `run(raw_data: str) -> str` køres per request
- `AZUREML_MODEL_DIR` peger på model-mappen i containeren

**Batch Endpoints:**
- `BatchEndpoint` + `BatchDeployment` — compute er AmlCompute cluster
- `mini_batch_size` = antal filer per batch til `run()` (ikke antal rækker!)
- `output_action`: `APPEND_ROW` skriver predictions til en CSV
- Invoke returnerer et job-objekt — monitorér med `ml_client.jobs.get()`
- Batch er asynkront — online er synkront

**Huske-regel:** Online = real-time + managed compute + synkront. Batch = store datamængder + AmlCompute + asynkront.